# 01. HCG 스키마 — 인과 그래프를 객체로 다루기

Humm은 인과 그래프를 LLM 내부 표현에 맡기지 않고 **Humm Causal Graph(HCG)** 라는 명시적 객체로 저장한다. 노드와 엣지에는 출처·증거·신뢰도·상태가 붙고, 이 정보가 이후 검증(`humm.validate`)과 학습(`humm.gym`, `humm.fast`)의 입력이 된다.

이 노트북에서는 다음을 다룬다.

1. 노드·엣지·증거로 그래프를 만든다
2. 증거로부터 엣지 신뢰도(noisy-OR)와 상태(hypothesis → supported)를 계산한다
3. 사이클을 찾아 가장 약한 엣지를 반증(refuted) 처리한다
4. LLM 프롬프트용 텍스트로 직렬화하고 JSON으로 저장한다
5. bnlearn 정답 그래프를 HCG로 불러와 SHD·SID 지표를 계산한다

In [ ]:
import networkx as nx
import pandas as pd

from humm.bench import datasets as D, metrics as M
from humm.paths import GRAPHS
from humm.graph import HCG, Evidence, EvidenceKind, to_compact_json, to_edge_text, to_prompt_block

## 1. 그래프 만들기

마케팅 예시로 `계절 → 광고비 → 방문 → 매출`, `계절 → 매출` 구조를 만든다. 계절은 광고비와 매출의 공통 원인, 즉 교란 변수이다. 엣지마다 증거를 붙이는데, 증거 종류(`text`, `data`, `llm_prior`, `expert`, `experiment`)가 신뢰도 가중치를 결정한다.

In [ ]:
g = HCG(graph_id="sales-demo", assumptions=["계절 외의 숨은 공통 원인은 없다"])
for nid, label in [("season", "계절"), ("ad", "광고비"), ("visits", "방문"), ("sales", "매출")]:
    g.add_node(nid, label=label, dtype="continuous")

g.add_edge("season", "ad", sign="+", mechanism="성수기에 광고 예산을 늘린다",
           evidence=[Evidence(kind=EvidenceKind.EXPERT, score=0.9, ref="마케팅팀 인터뷰")])
g.add_edge("ad", "visits", sign="+", mechanism="노출이 늘면 방문이 는다",
           evidence=[Evidence(kind=EvidenceKind.TEXT, score=0.8, ref="report_07.pdf#p3"),
                     Evidence(kind=EvidenceKind.DATA, score=0.9, method="PC+BK")])
g.add_edge("visits", "sales", sign="+", mechanism="방문이 구매로 전환된다",
           evidence=[Evidence(kind=EvidenceKind.TEXT, score=0.7), Evidence(kind=EvidenceKind.LLM_PRIOR, score=0.9)])
g.add_edge("season", "sales", sign="+", mechanism="성수기에는 수요 자체가 크다",
           evidence=[Evidence(kind=EvidenceKind.LLM_PRIOR, score=0.6)])
g.is_dag()

## 2. 신뢰도와 상태

엣지 신뢰도는 증거들의 noisy-OR 결합 `1 − Π(1 − w_kind · score)` 이다. 서로 다른 종류의 증거가 2개 이상이면 `hypothesis`에서 `supported`로 올라간다. `verified`와 `refuted` 판정은 데이터 검정이 필요하므로 Step 4의 `humm.validate`가 맡는다.

In [ ]:
for e in g.edges:
    e.promote()
pd.DataFrame([{"edge": f"{e.src}→{e.dst}", "n_evidence": len(e.evidence), "kinds": sorted({x.kind.value for x in e.evidence}),
               "confidence": round(e.confidence, 3), "status": e.status.value} for e in g.edges])

## 3. 사이클 해소

텍스트에서 "매출이 늘면 광고비를 늘린다" 같은 역방향 진술이 추출되면 사이클이 생긴다. 시간 지연이 없는 사이클은 DAG 가정을 깨므로, 사이클마다 신뢰도가 가장 낮은 엣지를 `refuted`로 바꾼다. 반증된 엣지는 삭제하지 않고 기록으로 남긴다.

In [ ]:
g.add_edge("sales", "ad", sign="+", evidence=[Evidence(kind=EvidenceKind.LLM_PRIOR, score=0.3)])
print("DAG:", g.is_dag(), "| cycles:", g.cycles())
print("refuted:", g.break_cycles(), "| DAG:", g.is_dag())

## 4. 직렬화와 저장

LLM에 그래프를 넣을 때는 반증된 엣지를 뺀 엣지 리스트를 쓴다. `to_edge_text`는 사람이 읽기 좋은 형식, `to_compact_json`은 토큰을 아끼는 형식이다.

In [ ]:
print(to_prompt_block(g))
print()
print(to_edge_text(g, with_confidence=True))
print()
print(to_compact_json(g))

In [ ]:
path = g.save(GRAPHS / "sales-demo.json")
assert HCG.load(path) == g
path

## 5. 정답 그래프와 지표

bnlearn의 Asia 네트워크를 불러와 HCG로 바꾸고, 일부러 훼손한 그래프와 비교한다. **SHD**는 구조 차이(누락·추가·반전)를, **SID**는 틀린 그래프로 개입 효과를 계산했을 때 틀리는 변수 쌍의 수를 센다. 인과 추론에는 SID가 더 직접적인 지표이다.

In [ ]:
asia = HCG.from_networkx(D.load_bnlearn("asia"), "asia")
true = asia.to_networkx()

def edit(g, remove=(), add=()):
    h = g.copy()
    h.remove_edges_from(remove)
    h.add_edges_from(add)
    return h

variants = {
    "정답": true,
    "smoke→lung 반전": edit(true, remove=[("smoke", "lung")], add=[("lung", "smoke")]),
    "either→dysp 누락": edit(true, remove=[("either", "dysp")]),
    "빈 그래프": edit(true, remove=list(true.edges)),
}
pd.DataFrame({k: M.graph_report(true, v) for k, v in variants.items()}).T

반전 하나는 SHD 1이지만 SID는 그보다 크게 나온다. 엣지 하나의 방향이 틀리면 그 엣지를 지나는 여러 개입 효과의 조정 집합이 함께 틀어지기 때문이다. Step 5부터는 탐색 알고리즘이 만든 그래프를 이 지표로 평가한다.

## Mac으로 옮길 파일

실행한 노트북은 원본과 다른 이름으로 저장하고, `notebooks/`의 원본은 출력 없이 둔다.

| 파일 | 필수 여부 | 비고 |
|---|---|---|
| 이 노트북 | 필수 | `runs/<날짜>-gpu/` |
| `data/graphs/sales-demo.json` | 선택 | 시드 고정, Mac에서 재현 가능 |